# Blast-to-Mill block-model polygon preview

This notebook recreates the behaviour needed for the Design Update Plan plot:

1. Select a `pattern_id` from `blast_master.csv` with an `ipywidgets.Select` component.
2. Draw the selected blast polygon from the ordered blast-master points.
3. Match block-model rows from `block_model_copper.csv` using the block centroid `x/y` world-coordinate columns.
4. Render the matched blocks below the polygon boundary with alpha applied.
5. Generate a conventional **6 m burden × 7 m spacing** drill-hole pattern and remove collars that fall outside the selected polygon.

The plotting layer intentionally treats the block model as `world_xy`, matching the handoff code path that expects scoped rows with `x/y` columns. The matching rule is centroid-in-polygon, which mirrors the current workflow's `pointInPolygon(block, ring)` behaviour.

## Environment

The notebook requires `pandas`, `numpy`, `matplotlib`, and `ipywidgets`. Most JupyterLab environments already have the first three. If the select UI does not appear, install widgets in the notebook environment:

```python
%pip install ipywidgets
```

Place `blast_master.csv` and `block_model_copper.csv` in the same directory as this notebook. The loader also checks `/mnt/data` so it works directly in this generated workspace.

In [1]:
from pathlib import Path
import math
import re
from dataclasses import dataclass
from typing import Any

from IPython.display import Markdown, display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon as MplPolygon, Rectangle
from matplotlib.path import Path as MplPath

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
    WIDGETS_IMPORT_ERROR = None
except Exception as exc:  # graceful static fallback when widgets are not installed
    widgets = None
    WIDGETS_AVAILABLE = False
    WIDGETS_IMPORT_ERROR = exc

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


In [2]:
# File loading and basic validation.

REQUIRED_BLAST_COLUMNS = {"pattern_id", "point_id", "x", "y"}
REQUIRED_BLOCK_COLUMNS = {"x", "y"}


def locate_input_file(filename: str) -> Path:
    """Return the first matching file path from common notebook/data locations."""
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "data" / filename,
        Path("/mnt/data") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename!r}. Put it next to this notebook, in ./data, or update the path in locate_input_file()."
    )


blast_master_path = locate_input_file("blast_master.csv")
block_model_path = locate_input_file("block_model_copper.csv")

blast_master = pd.read_csv(blast_master_path)
block_model = pd.read_csv(block_model_path)

missing_blast = REQUIRED_BLAST_COLUMNS.difference(blast_master.columns)
missing_blocks = REQUIRED_BLOCK_COLUMNS.difference(block_model.columns)
if missing_blast:
    raise ValueError(f"blast_master.csv is missing required columns: {sorted(missing_blast)}")
if missing_blocks:
    raise ValueError(f"block_model_copper.csv is missing required columns: {sorted(missing_blocks)}")

# Normalise the columns used by the plot.
blast_master["pattern_id"] = blast_master["pattern_id"].astype(str)
blast_master["point_id"] = pd.to_numeric(blast_master["point_id"], errors="coerce")
for col in ["x", "y", "floor_rl"]:
    if col in blast_master.columns:
        blast_master[col] = pd.to_numeric(blast_master[col], errors="coerce")
for col in ["x", "y", "z_top_m", "z_bottom_m"]:
    if col in block_model.columns:
        block_model[col] = pd.to_numeric(block_model[col], errors="coerce")


def natural_key(value: Any):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", str(value))]


pattern_ids = sorted(blast_master["pattern_id"].dropna().unique().tolist(), key=natural_key)
DEFAULT_PATTERN_ID = "A185-001-01" if "A185-001-01" in pattern_ids else pattern_ids[0]

numeric_metric_columns = [
    col
    for col in block_model.columns
    if pd.api.types.is_numeric_dtype(block_model[col])
    and not pd.api.types.is_bool_dtype(block_model[col])
    and col not in {"x", "y", "x_m", "y_m", "latitude", "longitude", "z_top_m", "z_bottom_m"}
]
DEFAULT_BLOCK_METRIC = "cu_pct" if "cu_pct" in numeric_metric_columns else (numeric_metric_columns[0] if numeric_metric_columns else "x")

print(f"Loaded blast master: {blast_master_path}  rows={len(blast_master):,}  patterns={len(pattern_ids):,}")
print(f"Loaded block model:  {block_model_path}  rows={len(block_model):,}  columns={len(block_model.columns):,}")
print(f"Default pattern:     {DEFAULT_PATTERN_ID}")
print(f"Default block metric:{DEFAULT_BLOCK_METRIC}")


Loaded blast master: c:\Users\expg\Downloads\blast_master.csv  rows=517  patterns=27
Loaded block model:  c:\Users\expg\Downloads\block_model_copper.csv  rows=5,892  columns=37
Default pattern:     A185-001-01
Default block metric:cu_pct


In [3]:
# Geometry, matching, and drill-hole generation helpers.


def open_ring(xy: np.ndarray) -> np.ndarray:
    """Return a polygon ring without a duplicated closing point."""
    arr = np.asarray(xy, dtype=float)
    arr = arr[np.isfinite(arr).all(axis=1)]
    if len(arr) >= 2 and np.allclose(arr[0], arr[-1]):
        arr = arr[:-1]
    return arr


def closed_ring(xy: np.ndarray) -> np.ndarray:
    """Return a polygon ring with the first point appended as the closing point."""
    arr = open_ring(xy)
    if len(arr) == 0:
        return arr
    return np.vstack([arr, arr[0]])


def polygon_path(xy: np.ndarray) -> MplPath:
    return MplPath(open_ring(xy))


def polygon_area_m2(xy: np.ndarray) -> float:
    """Shoelace area with local coordinates to avoid large-coordinate cancellation."""
    ring = open_ring(xy)
    if len(ring) < 3:
        return 0.0
    origin = ring[0]
    local = closed_ring(ring - origin)
    x = local[:, 0]
    y = local[:, 1]
    return float(abs(0.5 * np.sum(x[:-1] * y[1:] - x[1:] * y[:-1])))


def polygon_centroid(xy: np.ndarray) -> np.ndarray:
    """Area-weighted centroid, falling back to the mean for degenerate polygons."""
    ring = open_ring(xy)
    if len(ring) < 3:
        return np.nanmean(ring, axis=0)
    origin = ring[0]
    local = closed_ring(ring - origin)
    x = local[:, 0]
    y = local[:, 1]
    cross = x[:-1] * y[1:] - x[1:] * y[:-1]
    twice_area = np.sum(cross)
    if abs(twice_area) < 1e-9:
        return np.nanmean(ring, axis=0)
    cx = np.sum((x[:-1] + x[1:]) * cross) / (3 * twice_area)
    cy = np.sum((y[:-1] + y[1:]) * cross) / (3 * twice_area)
    return np.array([cx, cy]) + origin


def polygon_bounds(xy: np.ndarray) -> dict[str, float]:
    ring = open_ring(xy)
    return {
        "min_x": float(np.nanmin(ring[:, 0])),
        "max_x": float(np.nanmax(ring[:, 0])),
        "min_y": float(np.nanmin(ring[:, 1])),
        "max_y": float(np.nanmax(ring[:, 1])),
    }


def get_pattern_points(pattern_id: str) -> tuple[pd.DataFrame, np.ndarray]:
    pattern_rows = (
        blast_master.loc[blast_master["pattern_id"].astype(str) == str(pattern_id)]
        .sort_values("point_id", kind="mergesort")
        .copy()
    )
    if pattern_rows.empty:
        raise KeyError(f"Pattern {pattern_id!r} was not found in blast_master.csv")
    xy = open_ring(pattern_rows[["x", "y"]].to_numpy(dtype=float))
    if len(xy) < 3:
        raise ValueError(f"Pattern {pattern_id!r} has fewer than three valid polygon points")
    return pattern_rows, xy


def median_grid_step(values: pd.Series, fallback: float = 10.0) -> float:
    vals = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if vals.size < 2:
        return fallback
    unique = np.unique(np.round(vals, 6))
    diffs = np.diff(np.sort(unique))
    diffs = diffs[diffs > 1e-6]
    if diffs.size == 0:
        return fallback
    return float(np.median(diffs))


BLOCK_WIDTH_M = median_grid_step(block_model["x"], fallback=10.0)
BLOCK_HEIGHT_M = median_grid_step(block_model["y"], fallback=10.0)


def select_blocks_under_polygon(
    blocks_df: pd.DataFrame,
    polygon_xy: np.ndarray,
    floor_rl: float | None = None,
    match_bench: bool = True,
    boundary_radius_m: float = 1e-7,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Return block rows whose x/y centroid falls inside the selected polygon.

    The optional bench match keeps rows whose z_bottom_m equals the polygon floor RL.
    In the provided sample data all blocks and patterns are on RL185-200 / floor_rl=185.
    """
    bounds = polygon_bounds(polygon_xy)
    finite_xy = np.isfinite(blocks_df["x"]) & np.isfinite(blocks_df["y"])
    bbox_mask = (
        finite_xy
        & blocks_df["x"].between(bounds["min_x"], bounds["max_x"])
        & blocks_df["y"].between(bounds["min_y"], bounds["max_y"])
    )

    bench_mask = pd.Series(True, index=blocks_df.index)
    bench_filter_applied = False
    if match_bench and floor_rl is not None and np.isfinite(floor_rl) and "z_bottom_m" in blocks_df.columns:
        bench_mask = np.isclose(pd.to_numeric(blocks_df["z_bottom_m"], errors="coerce"), float(floor_rl), atol=0.1)
        bench_mask = pd.Series(bench_mask, index=blocks_df.index)
        bench_filter_applied = True

    candidate_index = blocks_df.index[bbox_mask & bench_mask]
    candidate_points = blocks_df.loc[candidate_index, ["x", "y"]].to_numpy(dtype=float)
    inside_values = polygon_path(polygon_xy).contains_points(candidate_points, radius=boundary_radius_m)

    inside_mask = pd.Series(False, index=blocks_df.index)
    inside_mask.loc[candidate_index] = inside_values
    matched = blocks_df.loc[inside_mask].copy()
    if "block_id" not in matched.columns:
        matched["block_id"] = matched.index.astype(str)

    diagnostics = {
        "bbox_candidates": int(len(candidate_index)),
        "matched_blocks": int(len(matched)),
        "bench_filter_applied": bench_filter_applied,
        "floor_rl": None if floor_rl is None or not np.isfinite(floor_rl) else float(floor_rl),
    }
    return matched, diagnostics


@dataclass
class FaceBasis:
    face_edge_index: int
    face_start: np.ndarray
    face_end: np.ndarray
    spacing_axis: np.ndarray
    burden_axis: np.ndarray


def longest_edge_face_basis(polygon_xy: np.ndarray, burden_m: float) -> FaceBasis:
    """Choose the longest polygon edge as the free face and point the burden axis into the polygon."""
    ring = open_ring(polygon_xy)
    path = polygon_path(ring)
    centroid = polygon_centroid(ring)
    best = None
    for i in range(len(ring)):
        start = ring[i]
        end = ring[(i + 1) % len(ring)]
        length = float(np.linalg.norm(end - start))
        if length > 1e-9 and (best is None or length > best[3]):
            best = (i, start, end, length)
    if best is None:
        raise ValueError("Could not find a valid polygon edge for drill-hole orientation")

    edge_index, start, end, length = best
    spacing_axis = (end - start) / length
    candidate_normal = np.array([-spacing_axis[1], spacing_axis[0]])
    midpoint = (start + end) / 2
    probe = max(float(burden_m), 1.0) * 0.25
    forward_inside = path.contains_point(midpoint + candidate_normal * probe, radius=1e-7)
    reverse_inside = path.contains_point(midpoint - candidate_normal * probe, radius=1e-7)
    if forward_inside and not reverse_inside:
        burden_axis = candidate_normal
    elif reverse_inside and not forward_inside:
        burden_axis = -candidate_normal
    else:
        burden_axis = candidate_normal if float(np.dot(centroid - midpoint, candidate_normal)) >= 0 else -candidate_normal

    return FaceBasis(edge_index, start, end, spacing_axis, burden_axis)


def arange_inclusive(start: float, stop: float, step: float) -> list[float]:
    if step <= 0 or stop < start:
        return []
    count = int(math.floor((stop - start) / step)) + 1
    values = [start + i * step for i in range(count)]
    # Include the end if it is very close to a grid point.
    if values and stop - values[-1] > step * 0.98:
        values.append(values[-1] + step)
    return values


def generate_drill_holes(
    polygon_xy: np.ndarray,
    burden_m: float = 6.0,
    spacing_m: float = 7.0,
    staggered: bool = True,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Generate a conventional burden x spacing drill pattern and clip collars outside the polygon."""
    burden_m = max(float(burden_m), 0.1)
    spacing_m = max(float(spacing_m), 0.1)
    ring = open_ring(polygon_xy)
    path = polygon_path(ring)
    basis = longest_edge_face_basis(ring, burden_m)

    rel = ring - basis.face_start
    u_values = rel @ basis.spacing_axis
    v_values = rel @ basis.burden_axis
    min_u, max_u = float(np.min(u_values)), float(np.max(u_values))
    min_v, max_v = float(np.min(v_values)), float(np.max(v_values))

    holes = []
    candidate_count = 0
    excluded_outside = 0
    row_index = 0
    for v in arange_inclusive(min_v + burden_m / 2, max_v, burden_m):
        stagger_offset = spacing_m / 2 if staggered and row_index % 2 == 1 else 0.0
        column_index = 0
        for u in arange_inclusive(min_u + spacing_m / 2 + stagger_offset, max_u, spacing_m):
            candidate_count += 1
            collar = basis.face_start + basis.spacing_axis * u + basis.burden_axis * v
            if not path.contains_point(collar, radius=1e-7):
                excluded_outside += 1
                column_index += 1
                continue
            holes.append(
                {
                    "hole_id": f"H{len(holes) + 1:03d}",
                    "row": row_index + 1,
                    "column": column_index + 1,
                    "x": float(collar[0]),
                    "y": float(collar[1]),
                    "u_m": float(u),
                    "v_m": float(v),
                    "burden_m": burden_m,
                    "spacing_m": spacing_m,
                }
            )
            column_index += 1
        row_index += 1

    diagnostics = {
        "candidate_holes_before_boundary_clip": int(candidate_count),
        "excluded_outside_boundary": int(excluded_outside),
        "generated_holes_inside": int(len(holes)),
        "face_edge_index": int(basis.face_edge_index),
        "staggered": bool(staggered),
    }
    return pd.DataFrame(holes), diagnostics


print(f"Inferred block footprint: {BLOCK_WIDTH_M:g} m x {BLOCK_HEIGHT_M:g} m")


Inferred block footprint: 10 m x 10 m


In [4]:
# Plot rendering.

last_render: dict[str, Any] = {}


def _finite_metric_values(df: pd.DataFrame, metric: str) -> pd.Series:
    if metric not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_numeric(df[metric], errors="coerce")


def render_pattern(
    pattern_id: str = DEFAULT_PATTERN_ID,
    block_metric: str = DEFAULT_BLOCK_METRIC,
    block_alpha: float = 0.45,
    hole_burden_m: float = 6.0,
    hole_spacing_m: float = 7.0,
    staggered: bool = True,
    match_bench: bool = True,
    label_holes: bool = False,
):
    """Render selected polygon, matched block footprints, and clipped drill-hole collars."""
    pattern_rows, polygon_xy = get_pattern_points(pattern_id)
    floor_rl = None
    if "floor_rl" in pattern_rows.columns and pattern_rows["floor_rl"].notna().any():
        floor_rl = float(pattern_rows["floor_rl"].median())

    matched_blocks, block_diag = select_blocks_under_polygon(
        block_model,
        polygon_xy,
        floor_rl=floor_rl,
        match_bench=match_bench,
    )
    holes, hole_diag = generate_drill_holes(
        polygon_xy,
        burden_m=hole_burden_m,
        spacing_m=hole_spacing_m,
        staggered=staggered,
    )

    last_render.clear()
    last_render.update(
        {
            "pattern_id": pattern_id,
            "pattern_rows": pattern_rows,
            "polygon_xy": polygon_xy,
            "matched_blocks": matched_blocks,
            "holes": holes,
            "block_diagnostics": block_diag,
            "hole_diagnostics": hole_diag,
        }
    )

    fig, ax = plt.subplots(figsize=(11, 8.5))

    # 1) Block model layer: draw first, under the polygon and holes.
    if not matched_blocks.empty:
        half_w = BLOCK_WIDTH_M / 2
        half_h = BLOCK_HEIGHT_M / 2
        rectangles = [
            Rectangle((row.x - half_w, row.y - half_h), BLOCK_WIDTH_M, BLOCK_HEIGHT_M)
            for row in matched_blocks[["x", "y"]].itertuples(index=False)
        ]
        metric_values = _finite_metric_values(matched_blocks, block_metric)
        if metric_values.notna().any():
            collection = PatchCollection(
                rectangles,
                cmap="viridis",
                alpha=float(block_alpha),
                edgecolor="none",
                linewidth=0,
                zorder=1,
            )
            collection.set_array(metric_values.to_numpy(dtype=float))
            valid_values = metric_values.dropna()
            if valid_values.nunique() == 1:
                value = float(valid_values.iloc[0])
                collection.set_clim(value - 1e-9, value + 1e-9)
            # Visually clip the cells to the selected polygon so the polygon remains the controlling boundary.
            collection.set_clip_path(MplPolygon(polygon_xy, closed=True, transform=ax.transData))
            ax.add_collection(collection)
            cbar = fig.colorbar(collection, ax=ax, shrink=0.82, pad=0.012)
            cbar.set_label(block_metric)
        else:
            collection = PatchCollection(
                rectangles,
                facecolor="tab:cyan",
                alpha=float(block_alpha),
                edgecolor="none",
                linewidth=0,
                zorder=1,
            )
            collection.set_clip_path(MplPolygon(polygon_xy, closed=True, transform=ax.transData))
            ax.add_collection(collection)

    # 2) Polygon layer: boundary above blocks.
    ring = closed_ring(polygon_xy)
    ax.fill(ring[:, 0], ring[:, 1], facecolor="none", edgecolor="black", linewidth=2.4, zorder=4)
    ax.plot(ring[:, 0], ring[:, 1], color="black", linewidth=2.4, zorder=4)
    ax.scatter(polygon_xy[:, 0], polygon_xy[:, 1], s=16, color="black", zorder=5, label="Polygon vertices")

    # 3) Drill-hole collars: above everything else.
    if not holes.empty:
        ax.scatter(
            holes["x"],
            holes["y"],
            s=42,
            marker="o",
            facecolor="white",
            edgecolor="black",
            linewidth=0.9,
            zorder=6,
            label="Generated drill holes",
        )
        if label_holes and len(holes) <= 120:
            for row in holes.itertuples(index=False):
                ax.text(row.x, row.y, row.hole_id, fontsize=6, ha="center", va="center", zorder=7)

    pattern_type = pattern_rows["pattern_type"].dropna().iloc[0] if "pattern_type" in pattern_rows.columns and pattern_rows["pattern_type"].notna().any() else "unknown"
    area_m2 = polygon_area_m2(polygon_xy)
    ax.set_title(
        f"Pattern {pattern_id} — {pattern_type} | blocks={len(matched_blocks):,} | holes={len(holes):,} | area={area_m2:,.0f} m²",
        fontsize=13,
        pad=12,
    )
    ax.set_xlabel("x / easting")
    ax.set_ylabel("y / northing")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linestyle=":", linewidth=0.7, alpha=0.55)
    ax.ticklabel_format(useOffset=False, style="plain")

    # Keep the view focused on the selected polygon plus rendered geometry.
    xs = [polygon_xy[:, 0]]
    ys = [polygon_xy[:, 1]]
    if not matched_blocks.empty:
        xs.append(matched_blocks["x"].to_numpy(dtype=float))
        ys.append(matched_blocks["y"].to_numpy(dtype=float))
    if not holes.empty:
        xs.append(holes["x"].to_numpy(dtype=float))
        ys.append(holes["y"].to_numpy(dtype=float))
    all_x = np.concatenate(xs)
    all_y = np.concatenate(ys)
    span_x = max(float(np.nanmax(all_x) - np.nanmin(all_x)), 1.0)
    span_y = max(float(np.nanmax(all_y) - np.nanmin(all_y)), 1.0)
    pad = max(12.0, 0.08 * max(span_x, span_y))
    ax.set_xlim(float(np.nanmin(all_x) - pad), float(np.nanmax(all_x) + pad))
    ax.set_ylim(float(np.nanmin(all_y) - pad), float(np.nanmax(all_y) + pad))

    legend_handles = [
        Patch(facecolor="tab:cyan", alpha=float(block_alpha), edgecolor="none", label="Matched block footprint layer"),
        Line2D([0], [0], color="black", linewidth=2.4, label="Blast polygon boundary"),
        Line2D([0], [0], marker="o", color="black", markerfacecolor="white", linestyle="None", markersize=7, label="Generated drill holes"),
    ]
    ax.legend(handles=legend_handles, loc="upper right", frameon=True)

    plt.show()

    summary = pd.DataFrame(
        [
            {
                "pattern_id": pattern_id,
                "pattern_type": pattern_type,
                "floor_rl": floor_rl,
                "polygon_vertices": len(polygon_xy),
                "polygon_area_m2": round(area_m2, 1),
                "bbox_block_candidates": block_diag["bbox_candidates"],
                "matched_blocks": block_diag["matched_blocks"],
                "bench_filter_applied": block_diag["bench_filter_applied"],
                "block_metric": block_metric,
                "block_alpha": round(float(block_alpha), 2),
                "hole_burden_m": round(float(hole_burden_m), 2),
                "hole_spacing_m": round(float(hole_spacing_m), 2),
                "hole_candidates_before_clip": hole_diag["candidate_holes_before_boundary_clip"],
                "holes_removed_outside_polygon": hole_diag["excluded_outside_boundary"],
                "holes_rendered": hole_diag["generated_holes_inside"],
                "hole_face_edge_index": hole_diag["face_edge_index"],
            }
        ]
    )
    display(summary)


In [5]:
# Interactive Select-based UI.

if WIDGETS_AVAILABLE:
    control_style = {"description_width": "120px"}
    pattern_select = widgets.Select(
        options=pattern_ids,
        value=DEFAULT_PATTERN_ID,
        rows=min(14, max(6, len(pattern_ids))),
        description="Pattern ID",
        style=control_style,
        layout=widgets.Layout(width="300px"),
    )
    metric_dropdown = widgets.Dropdown(
        options=numeric_metric_columns,
        value=DEFAULT_BLOCK_METRIC,
        description="Block metric",
        style=control_style,
        layout=widgets.Layout(width="300px"),
    )
    alpha_slider = widgets.FloatSlider(
        value=0.45,
        min=0.05,
        max=0.90,
        step=0.05,
        description="Block alpha",
        readout_format=".2f",
        style=control_style,
        layout=widgets.Layout(width="300px"),
    )
    burden_input = widgets.FloatText(
        value=6.0,
        description="Burden m",
        style=control_style,
        layout=widgets.Layout(width="300px"),
    )
    spacing_input = widgets.FloatText(
        value=7.0,
        description="Spacing m",
        style=control_style,
        layout=widgets.Layout(width="300px"),
    )
    staggered_checkbox = widgets.Checkbox(value=True, description="Stagger alternate rows")
    bench_checkbox = widgets.Checkbox(value=True, description="Match floor RL to z_bottom_m")
    label_checkbox = widgets.Checkbox(value=False, description="Label holes")

    controls = widgets.HBox(
        [
            widgets.VBox([pattern_select]),
            widgets.VBox([metric_dropdown, alpha_slider, burden_input, spacing_input, staggered_checkbox, bench_checkbox, label_checkbox]),
        ]
    )
    output = widgets.interactive_output(
        render_pattern,
        {
            "pattern_id": pattern_select,
            "block_metric": metric_dropdown,
            "block_alpha": alpha_slider,
            "hole_burden_m": burden_input,
            "hole_spacing_m": spacing_input,
            "staggered": staggered_checkbox,
            "match_bench": bench_checkbox,
            "label_holes": label_checkbox,
        },
    )
    display(controls, output)
else:
    display(
        Markdown(
            "**ipywidgets is not available in this kernel, so the select component cannot be displayed.**\n\n"
            "Run `%pip install ipywidgets`, restart the kernel, and re-run this cell to enable the interactive selector. "
            "A static render of the default pattern is shown below."
        )
    )
    print(f"ipywidgets import error: {WIDGETS_IMPORT_ERROR}")
    render_pattern(DEFAULT_PATTERN_ID)


Output()

## Inspect the selected rows after rendering

The renderer stores the latest selection in `last_render`, so you can inspect or export the matched blocks and generated holes after changing the pattern selector.

In [6]:
# Examples:
last_render["matched_blocks"].head()
last_render["holes"].head()
last_render["matched_blocks"].to_csv("matched_blocks_for_selected_pattern.csv", index=False)
last_render["holes"].to_csv("generated_holes_for_selected_pattern.csv", index=False)
